# Sentiment Model Training

This notebook demonstrates training a sentiment analysis model from scratch using scikit-learn.

## Objectives:
1. Prepare training data
2. Build and train a TF-IDF + Logistic Regression model
3. Evaluate model performance
4. Save the trained model


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import pickle
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

## Step 1: Prepare Training Data

Create a balanced dataset with positive, negative, and neutral sentiment examples.

In [ ]:
# Define sentiment lexicons
positive_adjectives = [
    'amazing', 'great', 'excellent', 'wonderful', 'fantastic', 'awesome',
    'beautiful', 'perfect', 'incredible', 'brilliant', 'love', 'best',
    'superb', 'outstanding', 'impressive', 'exceptional', 'phenomenal'
]

negative_adjectives = [
    'terrible', 'awful', 'horrible', 'bad', 'waste', 'disappointing',
    'horror', 'useless', 'stupid', 'poor', 'awful', 'hate', 'worst'
]

adverbs = ['really', 'very', 'absolutely', 'truly', 'quite', 'fairly',
           'highly', 'extremely', 'incredibly', 'surprisingly']

nouns = ['product', 'service', 'experience', 'support', 'update',
         'interface', 'quality', 'delivery', 'team', 'help']

# Function to generate synthetic reviews
def generate_review(positive=True):
    adj = np.random.choice(positive_adjectives if positive else negative_adjectives)
    adv = np.random.choice(adverbs)
    noun = np.random.choice(nouns)
    templates = [
        f"I {adv} {adj} {noun}!",
        f"{adv.capitalize()} {adj} {noun}.",
        f"This {noun} is {adv} {adj}!",
        f"Love this {noun}! It is {adv} {adj}.",
        f"The {noun} was {adj} as expected.",
        f"So {adj}! I'm {adv} happy with this.",
        f"What an {adj} {noun}! Definitely recommend.",
        f"{adv.capitalize()} {adj} experience overall.",
        f"Amazing! The {noun} is {adj}.",
        f"Best {noun} ever! {adj} results."
    ]
    return np.random.choice(templates)

# Generate training data
print("Generating training dataset...")

# Generate positive reviews
positive_reviews = [generate_review(positive=True) for _ in range(500)]

# Generate negative reviews
negative_reviews = [generate_review(positive=False) for _ in range(500)]

# Generate neutral reviews
neutral_templates = [
    "It's okay, nothing special.",
    "Average experience.",
    "Could be better.",
    "Nothing to say.",
    "It's what it is.",
    "As expected.",
    "Standard stuff.",
    "Nothing special.",
    "Meh.",
    "Just fine."
]
neutral_reviews = [np.random.choice(neutral_templates) for _ in range(300)]

# Combine into DataFrame
df_train = pd.DataFrame({
    'text': positive_reviews + negative_reviews + neutral_reviews,
    'label': [1]*500 + [0]*500 + [2]*300  # 1=positive, 0=negative, 2=neutral
})

print(f"Dataset created with {len(df_train)} samples")
print(f"Positive: {(df_train['label'] == 1).sum()}")
print(f"Negative: {(df_train['label'] == 0).sum()}")
print(f"Neutral: {(df_train['label'] == 2).sum()}")
print(f"\nSample reviews:")
print(df_train['text'].head(10).tolist())

## Step 2: Text Preprocessing

Clean and preprocess the text data before training.

In [ ]:
def preprocess_text(text):
    """Preprocess text for sentiment analysis."""
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text, flags=re.MULTILINE)
    
    # Remove mentions and hashtags (optional)
    # text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove special characters (keep basic punctuation)
    text = re.sub(r'[^\w\s\.,!?\'"]', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
print("Preprocessing text...")
df_train['cleaned_text'] = df_train['text'].apply(preprocess_text)

# Show before/after comparison
sample = df_train.iloc[0]
print(f"Original: {sample['text']}")
print(f"Cleaned:  {sample['cleaned_text']}")

## Step 3: Train the Model

Build and train a TF-IDF + Logistic Regression pipeline.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df_train['cleaned_text'],
    df_train['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_train['label']
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

# Create and train the model
model = Pipeline([
    ('vectorizer', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.8
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42,
        solver='lbfgs',
        class_weight='balanced'
    ))
])

print("\nTraining model...")
model.fit(X_train, y_train)
print("Model training complete!")

## Step 4: Evaluate the Model

Test the model on the held-out test set.

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("="*50)
print("MODEL EVALUATION RESULTS")
print("="*50)
print(f"\nTest Set Accuracy: {accuracy:.4f}")
print(f"\nConfusion Matrix:")
print(f"  {'Predicted: Negative':^25} {'Predicted: Positive':^25} {'Predicted: Neutral':^25}")
print(f"  {'Actual: Negative'  :^25} {cm[0,0]:>5} {cm[0,1]:>5} {cm[0,2]:>5}")
print(f"  {'Actual: Positive'  :^25} {cm[1,0]:>5} {cm[1,1]:>5} {cm[1,2]:>5}")
print(f"  {'Actual: Neutral'    :^25} {cm[2,0]:>5} {cm[2,1]:>5} {cm[2,2]:>5}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive', 'Neutral']))

## Step 5: Try Some Predictions

Test the model with sample texts.

In [ ]:
test_texts = [
    "I absolutely love this product! It's amazing and wonderful!",
    "Terrible experience! Worst product ever, very disappointing!",
    "It's okay, nothing special. Average quality.",
    "Best purchase ever! Highly recommend to everyone!",
    "Waste of money! Horrible quality and terrible service."
]

for text in test_texts:
    pred = model.predict([text])[0]
    proba = model.predict_proba([text])[0]
    
    labels = ['Negative', 'Positive', 'Neutral']
    confidence = np.max(proba)
    label = labels[pred]
    
    print(f"Text: {text[:50]}...")
    print(f"Prediction: {label} ({confidence:.2%})")
    print("-"*50)

## Step 6: Save the Model

Save the trained model for later use.

In [ ]:
# Save model
model_path = 'models/sentiment_model.pkl'
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

# Save preprocessing function
with open('models/preprocessor.py', 'w') as f:
    f.write('def preprocess_text(text):\n')
    f.write('    text = text.lower()\n')
    f.write('    text = re.sub(r"http\S+|www\S+", "", text, flags=re.MULTILINE)\n')
    f.write('    text = re.sub(r"[^\w\s\.,!?\'"]", "", text)\n')
    f.write('    text = re.sub(r"\s+", " ", text).strip()\n')
    f.write('    return text\n')
print("Preprocessor saved to models/preprocessor.py")

print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)

# 🎉 Congratulations!

You've successfully trained a sentiment analysis model! The model is now ready to be used in the API.

## Next Steps:
1. Use the trained model in your FastAPI app
2. Deploy with Docker
3. Run the Streamlit dashboard
4. Push to GitHub
